# Phase 0-1: Naive Inference Baseline on GPU (Kaggle / Colab T4)

This notebook benchmarks our custom from-scratch LLaMA inference engine against Hugging Face on an NVIDIA T4 GPU.

### Objectives:
1. Load SmolLM-135M directly from `safetensors` on GPU (NVIDIA T4).
2. Verify numerical correctness against Hugging Face reference.
3. Benchmark Naive Inference (no KV cache) measuring:
   - **TTFT (Time to First Token)**
   - **Decode Throughput (tokens / second)**
   - **Peak GPU Memory Allocated (MB)**
4. Compare directly against Hugging Face `model.generate()`.

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# 2. Clone Repository or Install Dependencies
!pip install -q transformers safetensors accelerate
# If running directly inside repository, ensure python path includes src:
import sys, os
sys.path.insert(0, os.path.abspath("../src") if os.path.exists("../src") else "src")

In [ ]:
# 3. Load Models onto GPU
from transformers import AutoModelForCausalLM, AutoTokenizer
from mini_infer.loader import load_from_hf
from mini_infer.generate import generate

MODEL_ID = "HuggingFaceTB/SmolLM-135M"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

print(f"Loading {MODEL_ID} on {device} ({dtype})...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

custom_model, config = load_from_hf(MODEL_ID, device=device, dtype=dtype)
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device).eval()

print("Both models loaded successfully!")

In [ ]:
# 4. Quick Correctness Check on GPU
prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)

with torch.no_grad():
    custom_logits = custom_model(input_ids)
    hf_logits = hf_model(input_ids).logits

max_diff = (custom_logits - hf_logits).abs().max().item()
print(f"Max GPU Logit Difference: {max_diff:.6e} (Threshold: 1e-4)")
assert max_diff < 1e-4, "Logit mismatch on GPU!"

In [ ]:
# 5. GPU Benchmark Suite (Custom Engine vs Hugging Face)
import time

BENCHMARK_PROMPTS = [
    "The capital of France is",
    "def fibonacci(n):",
    "The quick brown fox jumps",
    "In mathematics, the Riemann hypothesis is",
    "Machine learning models require data to",
]
max_new_tokens = 32
num_runs = 3

# Warmup
_ = generate(custom_model, tokenizer, BENCHMARK_PROMPTS[0], max_new_tokens=4, device=device)
if device == "cuda":
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

custom_ttft, custom_tps, custom_lat = [], [], []
for _ in range(num_runs):
    for p in BENCHMARK_PROMPTS:
        if device == "cuda": torch.cuda.synchronize()
        out = generate(custom_model, tokenizer, p, max_new_tokens=max_new_tokens, temperature=0.0, device=device)
        if device == "cuda": torch.cuda.synchronize()
        custom_ttft.append(out.ttft_ms)
        custom_tps.append(out.tokens_per_second)
        custom_lat.append(out.total_time_ms)

peak_gpu_mem_custom = (torch.cuda.max_memory_allocated() / (1024**2)) if device == "cuda" else 0.0

# Hugging Face Benchmark
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

hf_ttft, hf_tps, hf_lat = [], [], []
for _ in range(num_runs):
    for p in BENCHMARK_PROMPTS:
        inp = tokenizer(p, return_tensors="pt")["input_ids"].to(device)
        plen = inp.shape[1]
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            f_out = hf_model(inp)
            _ = f_out.logits[:, -1, :].argmax(dim=-1)
            if device == "cuda": torch.cuda.synchronize()
            t_first = time.perf_counter()
            full_out = hf_model.generate(inp, max_new_tokens=max_new_tokens, do_sample=False)
            if device == "cuda": torch.cuda.synchronize()
            t_end = time.perf_counter()
        
        hf_ttft.append((t_first - t0) * 1000.0)
        dec_toks = max(0, full_out.shape[1] - plen - 1)
        dec_time = t_end - t_first
        hf_tps.append((dec_toks / dec_time) if dec_time > 0 else 0.0)
        hf_lat.append((t_end - t0) * 1000.0)

peak_gpu_mem_hf = (torch.cuda.max_memory_allocated() / (1024**2)) if device == "cuda" else 0.0

print("\n=================== GPU BENCHMARK RESULTS (T4) ===================")
print(f"{'Engine':<25} | {'TTFT (ms)':<12} | {'Tokens / sec':<14} | {'Total Time (ms)':<16} | {'Peak GPU Mem (MB)':<16}")
print("-" * 92)
print(f"{'Custom Naive Engine':<25} | {sum(custom_ttft)/len(custom_ttft):<12.2f} | {sum(custom_tps)/len(custom_tps):<14.2f} | {sum(custom_lat)/len(custom_lat):<16.2f} | {peak_gpu_mem_custom:<16.2f}")
print(f"{'Hugging Face (generate)':<25} | {sum(hf_ttft)/len(hf_ttft):<12.2f} | {sum(hf_tps)/len(hf_tps):<14.2f} | {sum(hf_lat)/len(hf_lat):<16.2f} | {peak_gpu_mem_hf:<16.2f}")